In [1]:
import pandas as pd
import requests

In [2]:
# Load sales_transactions.csv from the URL

# Load it into a pandas DataFrame

# Display first few rows

In [3]:
def analyze_sales_data(transactions):
    """
    transactions: list of dicts, each dict has keys "Product", "Sales", "Customer_ID"
    
    Returns a dict with:
      - "total_sales_per_product": dict mapping product -> total sales
      - "top_3_products": list of up to 3 product names, sorted descending by total sales
      - "unique_customer_count": int number of distinct customers
      - "filtered_transactions": list of transaction dicts whose Sales > average Sales
    """
    # Handle empty input
    if not transactions:
        return {
            "total_sales_per_product": {},
            "top_3_products": [],
            "unique_customer_count": 0,
            "filtered_transactions": []
        }
    
    # 1. Compute total sales per product
    total_per_product = {}
    for tx in transactions:
        prod = tx.get("Product")
        sales = tx.get("Sales", 0)
        if prod is None:
            continue
        total_per_product[prod] = total_per_product.get(prod, 0) + sales
    
    # 2. Identify top 3 best-selling products by total sales
    # Sort products by total sales descending
    sorted_products = sorted(
        total_per_product.items(), key=lambda kv: kv[1], reverse=True
    )
    # Extract just product names (keys). Up to 3
    top_3 = [prod for prod, tot in sorted_products[:3]]
    
    # 3. Count unique customers
    customer_ids = set()
    for tx in transactions:
        cid = tx.get("Customer_ID")
        if cid is not None:
            customer_ids.add(cid)
    unique_count = len(customer_ids)
    
    # 4. Filter transactions where Sales > average sales
    # First compute average sales over all transactions
    total_sales_all = 0
    count_sales = 0
    for tx in transactions:
        # Only include transactions where "Sales" is present and numeric
        s = tx.get("Sales")
        if s is not None:
            total_sales_all += s
            count_sales += 1
    # If there are no sales entries, no filtered transactions
    if count_sales == 0:
        avg_sales = 0
    else:
        avg_sales = total_sales_all / count_sales
    
    # Now filter
    filtered = []
    for tx in transactions:
        s = tx.get("Sales")
        if s is not None and s > avg_sales:
            filtered.append(tx)
    
    # Prepare output
    result = {
        "total_sales_per_product": total_per_product,
        "top_3_products": top_3,
        "unique_customer_count": unique_count,
        "filtered_transactions": filtered
    }
    return result


In [4]:
#Do NOT modify this function
import pandas as pd
import inspect

def test_analyze_sales_data():
    """Test analyze_sales_data function using actual sales transaction data."""

    # Load dataset from URL
    url = "https://gitlab.crio.do/root/me_jupyter_numpy_pandas_interview_prep_datasets/-/raw/master/sales_transactions.csv"
    df = pd.read_csv(url)

    # Convert to list of dictionaries
    transactions = df.to_dict(orient="records")

    # Run the function
    result = analyze_sales_data(transactions)

    # Assertions on result structure
    assert isinstance(result, dict), "Function should return a dictionary"
    assert "total_sales_per_product" in result, "Missing key: total_sales_per_product"
    assert "top_3_products" in result, "Missing key: top_3_products"
    assert "unique_customer_count" in result, "Missing key: unique_customer_count"
    assert "filtered_transactions" in result, "Missing key: filtered_transactions"

    # Check types
    assert isinstance(result["total_sales_per_product"], dict), "total_sales_per_product should be a dict"
    assert isinstance(result["top_3_products"], list), "top_3_products should be a list"
    assert isinstance(result["unique_customer_count"], int), "unique_customer_count should be an int"
    assert isinstance(result["filtered_transactions"], list), "filtered_transactions should be a list"

    # Logical checks
    assert len(result["top_3_products"]) <= 3, "top_3_products should contain up to 3 items"
    assert result["unique_customer_count"] == df["Customer_ID"].nunique(), "Incorrect unique customer count"

    # Average sales check
    avg_sales = df["Sales"].mean()
    filtered_df = pd.DataFrame(result["filtered_transactions"])
    assert all(filtered_df["Sales"] > avg_sales), "filtered_transactions should only contain rows with sales above average"

    print("✅ Test Passed: analyze_sales_data() works correctly with the dataset.")

#Run the test
test_analyze_sales_data()

✅ Test Passed: analyze_sales_data() works correctly with the dataset.
